2.3.1 Импорты, seed и device

In [2]:
!pip install torch
!pip install torchvision
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, random_split

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/113.8 MB ? eta -:--:--
   ---------------------------------------- 0.5/113.8 MB 938.8 kB/s eta 0:02:01
   ---------------------------------------- 0.5/113.8 MB 938.8 kB/s eta 0:02:01
   ---------------------------------------- 0.8/113.8 MB 959.4 kB/s eta 0:01:58
   ---------------------------------------- 1.0/113.8 MB 950.3 kB/s eta 0:01:59
   ---------------------------------------- 1.3/113.8 MB 968.0 kB/s eta 0:01:57
   ---------------------------------------- 1.3/113.8 MB 968.0 kB/s eta 0:01:57
    --------------------------------------- 1.6/113.8 MB 969.0 kB/s eta 0:01:56
    --------------------------------------- 1.8/113.8 MB 981.9 kB/s eta 0:01:54
    --------------------------------------- 2.1/113.8 MB 1.0 MB/s eta


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\korobochka_sahara\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\korobochka_sahara\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---- ----------------------------------- 0.5/4.3 MB 5.1 MB/s eta 0:00:01
   --------------------- ------------------ 2.4/4.3 MB 7.6 MB/s eta 0:00:01
   -------------------------------------- - 4.2/4.3 MB 8.3 MB/s eta 0:00:01
   ---------------------------------------- 4.3/4.3 MB 8.0 MB/s  0:00:00
cpu


2.3.2. Данные и DataLoader

In [10]:
transform = transforms.ToTensor()

dataset = torchvision.datasets.EMNIST(
    root="./data",
    split='balanced',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.EMNIST(
    root="./data",
    split='balanced',
    train=False,
    download=True,
    transform=transform
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

x, y = next(iter(train_loader))

print(x.shape)
print(y.shape)
print(x.min(), x.max())

100%|██████████| 562M/562M [00:59<00:00, 9.47MB/s] 


torch.Size([128, 1, 28, 28])
torch.Size([128])
tensor(0.) tensor(1.)


2.3.3. Модель MLP и цикл обучения

In [16]:
class MLP(nn.Module):

    def __init__(self, hidden_sizes=[256,128], num_classes=10, dropout=0.0, batchnorm=False):
        super().__init__()

        layers = []
        input_size = 28*28

        for h in hidden_sizes:
            layers.append(nn.Linear(input_size, h))

            if batchnorm:
                layers.append(nn.BatchNorm1d(h))

            layers.append(nn.ReLU())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            input_size = h

        layers.append(nn.Linear(input_size, num_classes))

        self.net = nn.Sequential(
            nn.Flatten(),
            *layers
        )

    def forward(self, x):
        return self.net(x)  

Функции обучения

In [17]:
def train_one_epoch(model, loader, optimizer, criterion):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for x, y in loader:

        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * x.size(0)

        preds = logits.argmax(dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss/total, correct/total

def evaluate(model, loader, criterion):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in loader:

            x, y = x.to(device), y.to(device)

            logits = model(x)

            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)

            preds = logits.argmax(1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss/total, correct/total

Цикл обучения

In [18]:
def train_model(model, train_loader, val_loader, optimizer, criterion, epochs):

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    for epoch in range(epochs):

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)

        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(epoch, train_loss, val_loss, train_acc, val_acc)

    return history

3.1. Часть A (S08): регуляризация и переобучение

E1 base

In [20]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0,
    batchnorm=False
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)
history_E1 = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=15
)

0 1.1549876812502002 0.698974175740641 0.6590979609929078 0.780540780141844
1 0.5989493399224383 0.5694568775224348 0.8018949468085106 0.8151152482269504
2 0.4916032473246256 0.5168000716266903 0.8319038120567376 0.8306737588652482
3 0.427828534803492 0.48590030162892445 0.8504986702127659 0.8386524822695035
4 0.3831164841744917 0.4734885537877996 0.8631427304964538 0.8371010638297872
5 0.34849446200309914 0.4728056493803119 0.8723515070921986 0.8401595744680851
6 0.32071881909319694 0.47444067151411207 0.8799534574468085 0.8462765957446808
7 0.29673054717111247 0.46938656441708826 0.886934840425532 0.8485372340425532
8 0.27739740685156894 0.480370413749776 0.8930851063829788 0.8470744680851063
9 0.2591655229845791 0.49035337926648187 0.8981937056737589 0.8496010638297873
10 0.24053104255639068 0.5338222102070531 0.9040558510638298 0.843927304964539
11 0.2266566258690036 0.5207601894300881 0.9080562943262411 0.8512854609929078
12 0.21141373844856912 0.5593959971099881 0.914372783687943

E2 Dropout

In [23]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0.3,
    batchnorm=False
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history_E2 = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=15
)

0 1.476353841555034 0.7461349794205199 0.5654920212765957 0.7619237588652482
1 0.8390731037931239 0.6030070904298876 0.7330230496453901 0.8028812056737589
2 0.7199025101272772 0.5489399105944532 0.7684064716312057 0.8190602836879433
3 0.647474416562006 0.5289799167754802 0.7876329787234042 0.823758865248227
4 0.6110385571388488 0.4950201274655389 0.7972185283687944 0.8320921985815602
5 0.5791399994640486 0.4771978792992044 0.8041666666666667 0.8362145390070922
6 0.5539115205724189 0.46254649597702296 0.8128213652482269 0.8421985815602837
7 0.5338806855340376 0.44990143429303 0.8186281028368795 0.8439716312056738
8 0.521320944052216 0.4509072236980952 0.8217641843971631 0.8464095744680851
9 0.5042867302683228 0.4491609807555557 0.826917109929078 0.8456117021276596
10 0.49343308700737376 0.4367138515127466 0.828125 0.8513741134751773


E3 BatchNorm

In [25]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0,
    batchnorm=True
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history_E3 = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=15
)

0 0.8998400394375443 0.5516335494974827 0.7504321808510638 0.8154698581560283
1 0.48839318152015093 0.4741263041259549 0.8338209219858156 0.8385638297872341
2 0.41014432118716815 0.44154302439791093 0.8545434397163121 0.8493351063829787
3 0.3623812791937632 0.4386185945348537 0.8679742907801419 0.8507535460992908
4 0.3300476699644792 0.4395919359322135 0.8764073581560283 0.8520390070921986
5 0.30227022251338825 0.43335056199249644 0.8857269503546099 0.8530585106382979
6 0.2782933222169572 0.43777312281283925 0.8920877659574468 0.85625
7 0.2608606462360274 0.4376698531157582 0.8981604609929078 0.8543882978723404
8 0.2432335707101416 0.447080604911696 0.9043661347517731 0.8525709219858156
9 0.2312883660408622 0.4493068554722671 0.9067375886524822 0.8568705673758865
10 0.2186228484760785 0.45817483021012434 0.9118351063829787 0.8550975177304965
11 0.20385299252068742 0.46950922912739695 0.9176307624113476 0.8519503546099291
12 0.19606256064371014 0.4814246247000728 0.9184064716312057 0.85

E4 EarlyStopping

In [32]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0.3,
    batchnorm=False
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_val_loss = float("inf")
patience = 4
counter = 0

history_E4 = {
    "train_loss": [],
    "val_loss": []
}

for epoch in range(20):

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    history_E4["train_loss"].append(train_loss)
    history_E4["val_loss"].append(val_loss)

    print(epoch, train_loss, val_loss, val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0

        torch.save(model.state_dict(), "artifacts/best_model.pt")

    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered")
        break

0 1.493918916096924 0.7520021076743484 0.7619237588652482
1 0.8472787463073189 0.6076629378271441 0.8031914893617021
2 0.7200910383927906 0.5458113270448455 0.8208333333333333
3 0.6546383019457472 0.5158261738770397 0.8253102836879432
4 0.6110334527407978 0.49184833409938405 0.8345301418439717
5 0.5836984554503827 0.48982563881163904 0.8321365248226951
6 0.5574090572113687 0.47517833155943145 0.838741134751773
7 0.53884478439676 0.46687405912588675 0.8411790780141843
8 0.5238636708428673 0.4541629989096459 0.8426861702127659
9 0.5044999801943488 0.4492937266403902 0.847872340425532
10 0.4973543243205294 0.4439116710043968 0.85
11 0.4842603711794454 0.44120954253994826 0.8500886524822695
12 0.4750351783443005 0.441516057481157 0.8509751773049645
13 0.46819670859803547 0.4373971811423065 0.8484929078014184
14 0.45859793800834225 0.43776408641050896 0.8527925531914894
15 0.4529012367023644 0.4350167506975485 0.8522606382978724
16 0.4494734829410594 0.43706061865421053 0.8502659574468086
1

3.2. Часть B (S09): LR, оптимизаторы, weight decay

**O1 (LR слишком большой)**

In [33]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0.3,
    batchnorm=False
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-1
)
history_O1 = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=6
)

0 6.180223353027452 3.8581827884024764 0.021531471631205674 0.020434397163120566
1 3.870114501317342 3.871208842595418 0.02111037234042553 0.02052304964539007
2 3.86523067900475 3.8596435817420907 0.020500886524822695 0.02101063829787234
3 3.8653516076135297 3.8677098003685053 0.021897163120567376 0.021365248226950354
4 3.998842650947841 3.861389232865462 0.021453900709219858 0.021586879432624113
5 3.8648165865147366 3.867367192532154 0.02091090425531915 0.020257092198581562


**O2 (LR слишком маленький)**

In [34]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0.3,
    batchnorm=False
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-5
)
history_O2 = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=6
)

0 3.8232486653835216 3.752998124115856 0.04035904255319149 0.09720744680851064
1 3.612434383148843 3.3546121965908835 0.08939494680851064 0.1840868794326241
2 3.214465978635964 2.853022800269702 0.16914893617021276 0.3519503546099291
3 2.862966943294444 2.4882590060538434 0.23397606382978722 0.4192375886524823
4 2.6220012289412478 2.2506244808224074 0.28075132978723405 0.4583776595744681
5 2.454539386600467 2.0855695548632465 0.31491578014184396 0.48860815602836877


**O3 (SGD+momentum + weight decay)**

In [35]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0.3,
    batchnorm=False
).to(device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=1e-4
)
history_O3 = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=12
)

0 2.9318473398262728 1.482495019790974 0.22446808510638297 0.5778368794326241
1 1.4060225652464737 0.9531008431252013 0.5817597517730496 0.712854609929078
2 1.0648442998845526 0.7690661466713493 0.6718639184397163 0.7564716312056737
3 0.9158042409741287 0.6803574873200545 0.7131870567375886 0.7820035460992908
4 0.820669922084673 0.6363856955622951 0.7402593085106383 0.7910460992907802
5 0.7638076247475671 0.5867279663153574 0.7546431737588652 0.8067819148936171
6 0.7108617059727933 0.560651791433916 0.7698581560283688 0.8134308510638298
7 0.6753216736282862 0.5354657157938532 0.779864804964539 0.818927304964539
8 0.6424808450201724 0.5171983471998932 0.788386524822695 0.8271719858156028
9 0.6218705484207641 0.5044833636452966 0.795633865248227 0.8318705673758865
10 0.6017191391041938 0.4967288390964481 0.8001218971631205 0.8331117021276596
11 0.5804486633615291 0.4826718098728369 0.8046431737588653 0.8370124113475177


4. Артефакты

In [36]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

# curves_best.png

plt.figure()

plt.plot(history_E4["train_loss"], label="train loss")
plt.plot(history_E4["val_loss"], label="val loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Best model training curves")
plt.legend()

plt.savefig("artifacts/figures/curves_best.png")
plt.close()


# curves_lr_extremes.png

plt.figure()

plt.plot(history_O1["train_loss"], label="O1 large LR")
plt.plot(history_O2["train_loss"], label="O2 small LR")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning rate extremes")
plt.legend()

plt.savefig("artifacts/figures/curves_lr_extremes.png")
plt.close()


# best_config.json

best_config = {
    "dataset": "EMNIST",
    "seed": 42,
    "hidden_sizes": [512,256,128],
    "activation": "ReLU",
    "dropout": 0.3,
    "batchnorm": False,
    "optimizer": "Adam",
    "lr": 1e-3
}

with open("artifacts/best_config.json", "w") as f:
    json.dump(best_config, f, indent=4)


# runs.csv

runs = [
["E1","EMNIST",42,"512-256-128 ReLU","Adam",1e-3,0,0,15,0.851,0.469],
["E2","EMNIST",42,"512-256-128 + Dropout(0.3)","Adam",1e-3,0,0,11,0.851,0.436],
["E3","EMNIST",42,"512-256-128 + BatchNorm","Adam",1e-3,0,0,15,0.856,0.433],
["E4","EMNIST",42,"Dropout + EarlyStopping","Adam",1e-3,0,0,len(history_E4["train_loss"]),0.857,best_val_loss],
["O1","EMNIST",42,"same as E4","Adam",1e-1,0,0,6,None,None],
["O2","EMNIST",42,"same as E4","Adam",1e-5,0,0,6,None,None],
["O3","EMNIST",42,"same as E4","SGD",0.01,0.9,1e-4,12,None,None]
]

columns = [
"experiment_id","dataset","seed","model_summary",
"optimizer","lr","momentum","weight_decay",
"epochs_trained","best_val_accuracy","best_val_loss"
]

df = pd.DataFrame(runs, columns=columns)

df.to_csv("artifacts/runs.csv", index=False)